In [1]:
import sys 
import os 

sys.path.append(os.path.join('..', '..'))

In [2]:
from pathlib import Path
import pandas as pd
import requests
import zipfile
import pandas as pd 
import re 
import unicodedata

from config.bairro_cleaning_config import MAPA_BAIRRO, BAIRROS_OFICIAIS

In [3]:
data_dir = Path("../../data/ibge2010")
raw_dir = data_dir / "raw"

Vamos realizar o download dos três arquivos `.zip` necessários para a extração dos dados socioeconômicos:

- `censo2010_pr.zip`: contém os dados do censo demográfico de 2010 referentes ao estado do Paraná e possui os arquivos descritos em `DOCUMENTACAO_PARA_EXTRACAO.md`.

- `doc2010.zip`: contém a documentação técnica e o dicionário de dados. 

In [4]:
links = {
    "censo2010_pr": "https://ftp.ibge.gov.br/Censos/Censo_Demografico_2010/Resultados_do_Universo/Agregados_por_Setores_Censitarios/PR_20231030.zip",
    "doc2010": "https://ftp.ibge.gov.br/Censos/Censo_Demografico_2010/Resultados_do_Universo/Agregados_por_Setores_Censitarios/Documentacao_Agregado_dos_Setores_2010_20231030.zip",
}

In [5]:
for nome, url in links.items():
    destino = raw_dir / f"{nome}.zip"

    if not destino.exists():
        print(f"Baixando {nome}...")
        r = requests.get(url)
        r.raise_for_status()
        destino.write_bytes(r.content)

Baixando censo2010_pr...


Agora, precisamos extrair os arquivos zipados (https://docs.python.org/pt-br/3/library/zipfile.html):

In [6]:
for zip_path in raw_dir.glob("*.zip"):
    pasta_saida = raw_dir / zip_path.stem
    pasta_saida.mkdir(exist_ok=True)
    
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(pasta_saida)
    
    print(f"Extraído: {zip_path.name}")

Extraído: doc2010.zip
Extraído: censo2010_pr.zip


In [7]:
zip_path.unlink()

Agora utilizaremos o arquivo `Basico_PR.csv`. Esse arquivo é fundamental para a análise, pois contém a relação entre os setores censitários e suas respectivas divisões territoriais, incluindo o município e o bairro ao qual cada setor pertence.

Por meio dele, será possível identificar:

- o código de cada setor censitário;
- o código do bairro correspondente;
- o nome do bairro ao qual o setor pertence.

Essa relação será utilizada como uma tabela de referência para conectar os dados socioeconômicos das demais tabelas do censo de 2010.

Como as informações de renda, alfabetização e população estão organizadas originalmente em nível de setor censitário, será necessário agrupar os setores pertencentes a um mesmo bairro para gerar indicadores socioeconômicos consolidados em nível de bairro.

In [8]:
pasta_censo = raw_dir / "censo2010_pr/Base informaçoes setores2010 universo PR" 

Definiremos o caminho para os arquivos:

- `Basico_PR.csv`: será utilizado para relacionar cada setor censitário ao respectivo bairro de Curitiba;
- `PessoaRenda.csv`: será utilizado para extrair os indicadores de renda da população por setor censitário;
- `ResponsavelRenda.csv`: será utilizado para extrair os indicadores de renda dos responsáveis pela moradia, mantendo coerência com os dados do pipeline do censo de 2022;
- `Pessoa01.csv`: será utilizado para obter a quantidade de pessoas alfabetizadas por setor censitário;
- `Pessoa13.csv`: será utilizado para obter a população total por faixa etária, necessária para calcular as taxas de alfabetização.
- `Domicilio01_PR.csv`: será utilizado para obter indicadores de qualidade de vida da população, como: a presença de banheiros sanitários nas moradias particulares, sistema de esgoto precário, ausência de rede geral de água, presença de lixo destinado de forma inadequada e domicílios improvisados.

In [9]:
arquivo_basico = pasta_censo / "CSV/Basico_PR.csv"
arquivo_pessoa_renda = list(pasta_censo.rglob("*PessoaRenda*.csv"))[0]
arquivo_responsavel_renda = list(pasta_censo.rglob("*ResponsavelRenda*.csv"))[0]
arquivo_alfabetizacao = list(pasta_censo.rglob("*Pessoa01*.csv"))[0]
arquivo_idade = list(pasta_censo.rglob("*Pessoa13*.csv"))[0]
arquivo_domicilio = list(pasta_censo.rglob("*Domicilio01*.csv"))[0]

In [10]:
df = pd.read_csv(arquivo_basico, sep=";", encoding="latin1", dtype=str)

In [11]:
df.columns

Index(['Cod_setor', 'Cod_Grandes Regiões', 'Nome_Grande_Regiao', 'Cod_UF',
       'Nome_da_UF ', 'Cod_meso', 'Nome_da_meso', 'Cod_micro', 'Nome_da_micro',
       'Cod_RM', 'Nome_da_RM', 'Cod_municipio', 'Nome_do_municipio',
       'Cod_distrito', 'Nome_do_distrito', 'Cod_subdistrito',
       'Nome_do_subdistrito', 'Cod_bairro', 'Nome_do_bairro', 'Situacao_setor',
       'V001', 'V002', 'V003', 'V004', 'V005', 'V006', 'V007', 'V008', 'V009',
       'V010', 'V011', 'V012', 'Unnamed: 32'],
      dtype='str')

Vamos filtrar o dataframe pela coluna `Cod_municipio` para encontrar somente os dados pertencentes a Curitiba. Para isso, filtraremos pela coluna `Cod_municipio` e utilizaremos o código do município de Curitiba.

Fonte: https://www.ibge.gov.br/cidades-e-estados/pr/curitiba.html

In [12]:
df_curitiba = df.loc[df['Cod_municipio'] == '4106902']

In [13]:
df_curitiba['Cod_municipio'].unique()

<StringArray>
['4106902']
Length: 1, dtype: str

Agora, precisamos extrair o código do setor, que diz a qual bairro pertence cada setor censitário. Assim, conseguiremos interligação com outras tableas por meio do `Cod_setor` e unificar as informações por bairro.

In [14]:
codigo_bairro = df_curitiba[["Nome_do_bairro", "Cod_bairro", "Nome_do_municipio", "Cod_municipio"]]

In [15]:
codigo_bairro.groupby(["Cod_bairro"], dropna=True).first()

,Nome_do_bairro,Nome_do_municipio,Cod_municipio
Cod_bairro,,,
4106902001,Centro,CURITIBA,4106902
4106902002,São Francisco,CURITIBA,4106902
4106902003,Centro Cívico,CURITIBA,4106902
4106902004,Alto da Glória,CURITIBA,4106902
4106902005,Alto da Rua XV,CURITIBA,4106902
...,...,...,...
4106902071,Campo de Santana,CURITIBA,4106902
4106902072,Ganchinho,CURITIBA,4106902
4106902073,Umbará,CURITIBA,4106902


In [16]:
setores_bairro_curitiba = (
    df_curitiba[["Cod_setor", "Cod_bairro", "Nome_do_bairro"]]
    .dropna(subset=["Cod_setor", "Cod_bairro"])
    .drop_duplicates()
)

In [17]:
setores_bairro_curitiba.head()

,Cod_setor,Cod_bairro,Nome_do_bairro
4143,410690205010001,4106902001,Centro
4144,410690205010002,4106902001,Centro
4145,410690205010003,4106902001,Centro
4146,410690205010004,4106902001,Centro
4147,410690205010005,4106902001,Centro


## 1. Extraindo informações de população por bairro

Utilizando a documentação oficial do censo de 2010, podemos extrair a população de cada bairro utilizando a base de dados `Basico_PR.csv`, extraindo os valores por bairro para a coluna `V002`. 

De acordo com a documentação, a coluna `V002` representa o número de moradores em domicílios particulares permanentes ou população residente em domicílios particulares permanentes, o que fornece uma boa aproximação da população total do bairro.



In [18]:
coluna_populacao = "V002"

In [19]:
df_curitiba[coluna_populacao] = pd.to_numeric(
    df[coluna_populacao],
    errors="coerce"
)

In [20]:
populacao_por_bairro = (
    df_curitiba
    .groupby(["Cod_bairro", "Nome_do_bairro"])[coluna_populacao]
    .sum(min_count=1)
    .reset_index(name="populacao_2010")
)

In [21]:
populacao_por_bairro

,Cod_bairro,Nome_do_bairro,populacao_2010
0,4106902001,Centro,36061.0
1,4106902002,São Francisco,5955.0
2,4106902003,Centro Cívico,4756.0
3,4106902004,Alto da Glória,5520.0
4,4106902005,Alto da Rua XV,8474.0
...,...,...,...
70,4106902071,Campo de Santana,27156.0
71,4106902072,Ganchinho,11175.0
72,4106902073,Umbará,18721.0
73,4106902074,Tatuquara,52208.0


In [22]:
populacao_por_bairro["populacao_2010"] = (
    populacao_por_bairro["populacao_2010"].astype("Int64")
)

In [23]:
pop_total_bairros = populacao_por_bairro["populacao_2010"].sum()

pop_total_bairros

np.int64(1744129)

## 2. Extraindo informações de renda

Para extrair informações relacionadas à renda das pessoas, utilizaremos o arquivo `PessoaRenda_PR.csv`.

Esse arquivo contém informações de rendimento nominal mensal das pessoas de 10 anos ou mais de idade, organizadas por setor censitário. Assim, iremos utilizar o código do setor censitário como chave para interligar esses dados em nível de bairro. 

As variáveis que serão extraídas são:

| Indicador | Variáveis usadas | Fórmula|
|-----------|------------------|--------|
| Pessoas sem rendimento | `V010` | `V010` / (`V020` * 100) |
| Pessoas com rendimento até 1 salário min. | `V001` + `V002` | (`V001` + `V002`)/(`V020`) * 100 |
| Pessoas com rendimento até 2 salários min. | `V001` + `V002` + `V003`| (`V001` + `V002` + `V003`) / (`V020`) * 100 |
| Pessoas com rendimento acima de 5 salários min. | `V006` + `V007` + `V008` + `V009` | (`V006` + `V007` + `V008` + `V009`) / (`V020`) * 100 |

Dicionário de variáveis:

- `V001`: pessoas de 10 anos ou mais de idade com rendimento nominal mensal de até $1/2$ salário mínimo;
- `V002`: pessoas de 10 anos ou mais de idade com rendimento nominal mensal de mais de $1/2$ a 1 salário mínimo;
- `V003`: pessoas de 10 anos ou mais de idade com rendimento nominal mensal de mais de 1 a 2 salários mínimos;
- `V006`: pessoas de 10 anos ou mais de idade com rendimento nominal mensal de mais de 5 a 10 salários mínimos;
- `V007`: pessoas de 10 anos ou mais de idade com rendimento nominal mensal de mais de 10 a 15 salários mínimos;
- `V008`: pessoas de 10 anos ou mais de idade com rendimento nominal mensal de mais de 15 a 20 salários mínimos;
- `V009`: pessoas de 10 anos ou mais de idade com rendimento nominal mensal de mais de 20 salários mínimos;
- `V020`: pessoas de 10 anos ou mais de idade com ou sem rendimento. 

In [24]:
pessoa_renda = pd.read_csv(
    arquivo_pessoa_renda,
    sep=";",
    encoding="latin1",
    dtype=str
)

In [25]:
renda_curitiba = pessoa_renda.merge(
    setores_bairro_curitiba,
    on="Cod_setor",
    how="inner"
)

In [26]:
variaveis_renda = [
    "V001",
    "V002", 
    "V003", 
    "V006", 
    "V007", 
    "V008",
    "V009", 
    "V010",
    "V020"
]

In [27]:
for coluna in variaveis_renda:
    renda_curitiba[coluna] = (
        renda_curitiba[coluna]
        .astype(str)
        .str.strip()
        .replace(["X", "", "nan"], pd.NA)
        .str.replace(",", ".", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )

In [28]:
renda_por_bairro = (
    renda_curitiba
    .groupby(["Cod_bairro", "Nome_do_bairro"], as_index=False)[variaveis_renda]
    .sum()
)

#### 2.1 Cálculo do percentual de pessoas sem rendimento

In [29]:
renda_por_bairro["pct_sem_rendimento"] = (
    renda_por_bairro["V010"] / renda_por_bairro["V020"]
) * 100

#### 2.2 Cálculo do percentual de pessoas com rendimento de até 1 salário mínimo

In [30]:
renda_por_bairro["pct_rendimento_ate_1_sm"] = (
    (renda_por_bairro["V001"] + renda_por_bairro["V002"]) /
    renda_por_bairro["V020"]
) * 100

#### 2.2 Cálculo do percentual de pessoas com rendimento de até 2 salário mínimo

In [31]:
renda_por_bairro["pct_rendimento_ate_2_sm"] = (
    (renda_por_bairro["V001"] + renda_por_bairro["V002"] + renda_por_bairro["V003"]) /
    renda_por_bairro["V020"]
) * 100

#### 2.2 Cálculo do percentual de pessoas com rendimento acima de 5 salários mínimos

In [32]:
renda_por_bairro["pct_rendimento_acima_5_sm"] = (
    (
        renda_por_bairro["V006"] +
        renda_por_bairro["V007"] +
        renda_por_bairro["V008"] +
        renda_por_bairro["V009"]
    ) / renda_por_bairro["V020"]
) * 100

In [33]:
renda_bairros_final = renda_por_bairro[
    [
        "Cod_bairro",
        "Nome_do_bairro",
        "V020",
        "pct_sem_rendimento",
        "pct_rendimento_ate_1_sm",
        "pct_rendimento_ate_2_sm",
        "pct_rendimento_acima_5_sm"
    ]
].rename(columns={
    "V020": "pessoas_10_anos_ou_mais"
})

In [34]:
renda_bairros_final

,Cod_bairro,Nome_do_bairro,pessoas_10_anos_ou_mais,pct_sem_rendimento,pct_rendimento_ate_1_sm,pct_rendimento_ate_2_sm,pct_rendimento_acima_5_sm
0,4106902001,Centro,35353.0,22.524255,5.722287,21.797302,27.788307
1,4106902002,São Francisco,5703.0,21.251973,6.382606,23.619148,26.424689
2,4106902003,Centro Cívico,4480.0,20.334821,3.950893,15.156250,40.446429
3,4106902004,Alto da Glória,5163.0,25.527794,3.137711,12.880108,39.996126
4,4106902005,Alto da Rua XV,7396.0,24.824229,4.732288,16.184424,35.667929
...,...,...,...,...,...,...,...
70,4106902071,Campo de Santana,21993.0,33.847133,14.295458,48.469968,1.309508
71,4106902072,Ganchinho,9087.0,35.105095,18.774073,49.741389,1.804776
72,4106902073,Umbará,15721.0,34.482539,13.141658,42.573628,3.606641
73,4106902074,Tatuquara,43116.0,34.701735,16.963540,50.166991,1.185175


Para manter coerência com os dados obtidos no censo de 2022, vamos também obter os dados relacionados a renda dos responsáveis pela moradia. Para isso, faremos:

In [35]:
responsavel_renda = pd.read_csv(
    arquivo_responsavel_renda,
    sep=";",
    encoding="latin1",
    dtype="str"
)

Assim como feito para 2022, vamos inserir uma metrica chamada `rendimento_medio_responsavel_sm`, que mede quantos salários mínimos em média os responsáveis por moradias particulares recebem em determinado bairro.

In [36]:
salario_minimo_2010 = 510

In [37]:
responsavel_renda_curitiba = responsavel_renda.merge(
    setores_bairro_curitiba,
    on="Cod_setor",
    how="inner"
)

In [38]:
variaveis_responsavel_renda = ["V020", "V021", "V022"]

- `V020`: total de responsáveis com ou sem rendimento por bairro;
- `V021`: total de responsáveis com rendimentos positivo;
- `V022`: soma do rendimento nominal mensal dos responsáveis.

In [39]:
for coluna in variaveis_responsavel_renda:
    responsavel_renda_curitiba[coluna] = (
        responsavel_renda_curitiba[coluna]
        .astype(str)
        .str.strip()
        .replace(["X", "", "nan"], pd.NA)
        .str.replace(",", ".", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )

In [40]:
responsavel_renda_por_bairro = (
    responsavel_renda_curitiba
    .groupby(["Cod_bairro", "Nome_do_bairro"], as_index=False)
    [variaveis_responsavel_renda]
    .sum()
)

In [41]:
responsavel_renda_por_bairro["resp_domicilios_particulares"] = (
      responsavel_renda_por_bairro["V020"]
  )

In [42]:
responsavel_renda_por_bairro["rendimento_medio_responsavel"] = (
    responsavel_renda_por_bairro["V022"] /
    responsavel_renda_por_bairro["V021"]
).where(responsavel_renda_por_bairro["V021"] > 0)

In [43]:
responsavel_renda_por_bairro["rendimento_medio_responsavel_sm"] = (
    responsavel_renda_por_bairro["rendimento_medio_responsavel"] /
    salario_minimo_2010
)

In [44]:
responsavel_renda_bairros_final = responsavel_renda_por_bairro[
    [
        "Cod_bairro",
        "Nome_do_bairro",
        "resp_domicilios_particulares",
        "rendimento_medio_responsavel",
        "rendimento_medio_responsavel_sm"
    ]
].copy()

In [45]:
renda_bairros_final = renda_bairros_final.merge( 
    responsavel_renda_bairros_final,
    on=["Cod_bairro", "Nome_do_bairro"],
    how="left",
    validate="1:1"
)

In [46]:
renda_bairros_final.head(3)

,Cod_bairro,Nome_do_bairro,pessoas_10_anos_ou_mais,pct_sem_rendimento,pct_rendimento_ate_1_sm,pct_rendimento_ate_2_sm,pct_rendimento_acima_5_sm,resp_domicilios_particulares,rendimento_medio_responsavel,rendimento_medio_responsavel_sm
0,4106902001,Centro,35353.0,22.524255,5.722287,21.797302,27.788307,17763.0,3775.695779,7.403325
1,4106902002,São Francisco,5703.0,21.251973,6.382606,23.619148,26.424689,2433.0,3530.811436,6.923160
2,4106902003,Centro Cívico,4480.0,20.334821,3.950893,15.156250,40.446429,2106.0,4869.421827,9.547886


### 3. Extraindo informações de alfabetização

Para calcular as taxas de alfabetização dos bairros de Curitiba iremos utilizar os arquivos:

- `Pessoa01_PR.csv`: utilizado para obter a quantidade de pessoas alfabetizadas por setor censitário;
- `Pessoa13_PR.csv`: utilizado para obter a população total por idade em cada setor censitário.

Assim, o arquivo `Pessoa01_PR.csv` informa a qunatidade de pessoas alfabetizadas, enquanto o arquivo `Pessoa13_PR.csv` permite obter o total de pessoas na mesma faixa etária, assim, conseguimos calcular a taxa de alfabetização para diferentes faixas etárias. Além disso, também utilizaremos a `setores_bairro_curitiba`, construído com o arquivo `Basico_PR.csv` para interligar esses resultados a cada um dos bairros.

`Pessoa01_PR.csv`: 

Nessa tabela, utilizaremos as variáveis:

- `V007`: pessoas alfabetizadas com 10 anos de idade;
  ...
- `V077`: pessoas alfabetizadas com 80 anos ou mais de idade.

`Pessoa13_PR.csv`:

- `V050`: pessoas com 10 anos de idade;
...
- `V134`: pessoas com 100 anos ou mais de idade.

In [47]:
alfabetizacao = pd.read_csv(
    arquivo_alfabetizacao,
    sep=";",
    encoding="latin1",
    dtype=str
)

In [48]:
idade = pd.read_csv(
    arquivo_idade,
    sep=";",
    encoding="latin1",
    dtype=str
)

In [49]:
alfabetizacao_curitiba = alfabetizacao.merge(
    setores_bairro_curitiba,
    on="Cod_setor",
    how="inner"
)

In [50]:
alfabetizacao_curitiba.head(3)

,Cod_setor,Situacao_setor,V001,V002,V003,V004,V005,V006,V007,V008,...,V079,V080,V081,V082,V083,V084,V085,Unnamed: 87,Cod_bairro,Nome_do_bairro
0,410690205010001,1,476,1,0,3,3,3,4,5,...,70,78,6,10,15,17,47,NaN,4106902001,Centro
1,410690205010002,1,559,0,3,1,0,1,2,2,...,81,63,19,7,29,17,98,NaN,4106902001,Centro
2,410690205010003,1,487,3,3,2,2,0,2,5,...,62,59,28,9,24,13,42,NaN,4106902001,Centro


In [51]:
idade_curitiba = idade.merge(
    setores_bairro_curitiba,
    on="Cod_setor",
    how="inner"
)

In [52]:
idade_curitiba.head(3)

,Cod_setor,Situacao_setor,V001,V002,V003,V004,V005,V006,V007,V008,...,V128,V129,V130,V131,V132,V133,V134,Unnamed: 136,Cod_bairro,Nome_do_bairro
0,410690205010001,1,506,505,220,73,41,56,4,4,...,0,0,0,0,0,0,0,NaN,4106902001,Centro
1,410690205010002,1,578,578,237,81,50,30,0,0,...,0,1,0,0,0,0,0,NaN,4106902001,Centro
2,410690205010003,1,502,492,229,62,37,35,0,1,...,1,0,1,0,0,0,0,NaN,4106902001,Centro


#### 3.1 Alfabetizados com 10 anos ou mais

In [53]:
alfabetizados_10mais = []

for i in range(7, 78):
    alfabetizados_10mais.append(f"V{i:03d}")

#### 3.2 Alfabetizados com 15 anos ou mais

In [54]:
alfabetizados_15mais = []

for i in range(12, 78):
    alfabetizados_15mais.append(f"V{i:03d}")

Criaremos uma lista contendo todas as colunas relacionadas às faixas etárias que serão utilizadas no cálculo das taxas de alfabetização:

- `alfabetizados_10mais`;
- `alfabetizados_15mais`

Como algumas colunas poderão estar presentes em mais de uma faixa etária, utilizaremos `set()` para remover valores duplicados, garantindo que cada coluna apareça apenas uma vez na lista final.

In [55]:
colunas_alfabetizacao = list(set(
    alfabetizados_10mais +
    alfabetizados_15mais
))

#### 3.3 População com 10 anos ou mais

In [56]:
pop_10mais = []

for i in range(44, 135):
    pop_10mais.append(f"V{i:03d}")

#### 3.6 População com 15 anos ou mais

In [57]:
pop_15mais = []

for i in range(49, 135):
    pop_15mais.append(f"V{i:03d}")

Criaremos uma lista contendo todas as colunas relacionadas às faixas etárias que serão utilizadas no cálculo das taxas de alfabetização:

- `pop_10mais`;
- `pop_15mais`

Como algumas colunas poderão estar presentes em mais de uma faixa etária, utilizaremos `set()` para remover valores duplicados, garantindo que cada coluna apareça apenas uma vez na lista final.

In [58]:
colunas_idade = list(set(
    pop_10mais +
    pop_15mais
))

### Limpeza de `alfabetizado_curitiba` e `idade_curitiba`:

- removeremos espaços em branco;
- substituiremos valores inválidos ou ausentes ("X" e "nan") por valores nulos;
- ajustaremos o separador decimal, subtituindo vírgulas por pontos;
- conveteremos os valores para formato numérico.

In [59]:
for coluna in colunas_alfabetizacao:
    alfabetizacao_curitiba[coluna] = (
        alfabetizacao_curitiba[coluna]
        .astype(str)
        .str.strip()
        .replace(["X", "", "nan"], pd.NA)
        .str.replace(",", ".", regex=False)
    )
    alfabetizacao_curitiba[coluna] = pd.to_numeric(
        alfabetizacao_curitiba[coluna],
        errors="coerce"
    )

In [60]:
for coluna in colunas_idade:
    idade_curitiba[coluna] = (
        idade_curitiba[coluna]
        .astype(str)
        .str.strip()
        .replace(["X", "", "nan"], pd.NA)
        .str.replace(",", ".", regex=False)
    )
    idade_curitiba[coluna] = pd.to_numeric(
        idade_curitiba[coluna],
        errors="coerce"
    )

In [61]:
alfabetizacao_curitiba["alfabetizados_10mais"] = (
    alfabetizacao_curitiba[alfabetizados_10mais].sum(axis=1)
)

alfabetizacao_curitiba["alfabetizados_15mais"] = (
    alfabetizacao_curitiba[alfabetizados_15mais].sum(axis=1)
)

In [62]:
idade_curitiba["pop_10mais"] = idade_curitiba[pop_10mais].sum(axis=1)

idade_curitiba["pop_15mais"] = idade_curitiba[pop_15mais].sum(axis=1)

In [63]:
alfabetizacao_por_bairro = (
    alfabetizacao_curitiba
    .groupby(["Cod_bairro", "Nome_do_bairro"], as_index=False)
    [["alfabetizados_10mais", "alfabetizados_15mais"]]
    .sum()
)

In [64]:
idade_por_bairro = (
    idade_curitiba
    .groupby(["Cod_bairro", "Nome_do_bairro"], as_index=False)
    [["pop_10mais", "pop_15mais"]]
    .sum()
)

Agora, realizaremos o merge por bairro dos dados calculados:

In [65]:
alfabetizacao_bairros = alfabetizacao_por_bairro.merge(
    idade_por_bairro,
    on=["Cod_bairro", "Nome_do_bairro"],
    how="inner"
)

A quantidade de pessoas analfabetas será calculada fazendo:

$$
    \text{Pessoas analfabetas} = \text{Quantidade de pessoas} - \text{Pessoas alfabetizadas}
$$

In [66]:
alfabetizacao_bairros["pct_alfabetizacao_10mais"] = (
    alfabetizacao_bairros["alfabetizados_10mais"] /
    alfabetizacao_bairros["pop_10mais"]
) * 100

In [67]:
alfabetizacao_bairros["pct_analfabetismo_10mais"] = (
    100 - alfabetizacao_bairros["pct_alfabetizacao_10mais"]
)

In [68]:
alfabetizacao_bairros["pct_alfabetizacao_15mais"] = (
    alfabetizacao_bairros["alfabetizados_15mais"] /
    alfabetizacao_bairros["pop_15mais"]
) * 100

In [69]:
alfabetizacao_bairros["analfabetos_15mais"] = (
    alfabetizacao_bairros["pop_15mais"]
    - alfabetizacao_bairros["alfabetizados_15mais"]
)

In [70]:
alfabetizacao_bairros["pct_analfabetismo_15mais"] = (
    100 - alfabetizacao_bairros["pct_alfabetizacao_15mais"]
)

In [71]:
alfabetizacao_bairros_final = alfabetizacao_bairros[
    [
        "Cod_bairro",
        "Nome_do_bairro",
        "pop_10mais",
        "alfabetizados_10mais",
        "pct_alfabetizacao_10mais",
        "pct_analfabetismo_10mais",
        "pop_15mais",
        "alfabetizados_15mais",
        "analfabetos_15mais",
        "pct_alfabetizacao_15mais",
        "pct_analfabetismo_15mais"
    ]
].copy()

In [72]:
alfabetizacao_bairros_final.head()

,Cod_bairro,Nome_do_bairro,pop_10mais,alfabetizados_10mais,pct_alfabetizacao_10mais,pct_analfabetismo_10mais,pop_15mais,alfabetizados_15mais,analfabetos_15mais,pct_alfabetizacao_15mais,pct_analfabetismo_15mais
0,4106902001,Centro,35353.0,35138.0,99.391848,0.608152,34333.0,34127.0,206.0,99.399994,0.600006
1,4106902002,São Francisco,5703.0,5678.0,99.561634,0.438366,5485.0,5463.0,22.0,99.598906,0.401094
2,4106902003,Centro Cívico,4480.0,4469.0,99.754464,0.245536,4319.0,4309.0,10.0,99.768465,0.231535
3,4106902004,Alto da Glória,5163.0,5144.0,99.631997,0.368003,4955.0,4938.0,17.0,99.656912,0.343088
4,4106902005,Alto da Rua XV,7396.0,7363.0,99.553813,0.446187,7070.0,7040.0,30.0,99.575672,0.424328


### 4. Extraindo informações quanto ao saneamento básico

Para calcular informações relacionadas ao saneamento básico dos bairro de Curitiba, utilizaremos o arquivo:

- `Domicilio01_PR.csv`

Nessa tabela, utilizaremos as variáveis:

- `V013`: domicílios particulares permanentes com abastecimento de água de poço ou nascente na propriedade;
- `V014`: domicílios particulares com abastecimento de água da chuva armazenada em cisterna;
- `V015`: domicílios particulares com outra forma de abastecimento de água;
- `V023`: domicílios particulares permanentes sem banheiro de uso exclusivo dos moradores e nem sanitário;
- `V019`: domicílios particulares permanentes com banheiro de uso exclusivo dos moradores ou sanitário e esgotamento sanitário via fossa rudimentar;
- `V020`: domicílios particulares permanentes com banheiro de uso exclusivo dos moradores ou sanitário e esgotamento sanitário via vala;
- `V021`: domicílios particulares permanentes, com banheiro de uso exclusivo dos moradores ou sanitário e esgotamento sanitário via rio, lago ou mar;
- `V022`: domicílios particulares permanentes com banheiro de uso exclusivo dos moradores ou sanitário e esgotamento sanitário via outro escoadouro;
- `V023`: domicílios particulares permanentes sem banheiro de uso exclusivo dos moradores e nem sanitário;
- `V038`: domicílios particulares permanentes com lixo queimado na propriedade;
- `V039`: domicílios particulares permanentes com lixo enterrado na propriedade;
- `V040`: domicílios particulares permanentes com lixo jogado em terreno baldio ou logradouro;
- `V041`: domicílios particulares permanentes com lixo jogado em rio, lago ou mar;
- `V042`: domicílios particulares permanentes com outro destino do lixo.

In [73]:
domicilio = pd.read_csv(
    arquivo_domicilio,
    sep=";",
    encoding="latin1",
    dtype=str
)

In [74]:
saneamento_curitiba = domicilio.merge(
    setores_bairro_curitiba,
    on="Cod_setor",
    how="inner"
)

In [75]:
variaveis_saneamento = [
    "V002",
    "V013",
    "V014",
    "V015",
    "V019",
    "V020",
    "V021",
    "V022",
    "V023",
    "V038",
    "V039",
    "V040",
    "V041",
    "V042"
]

In [76]:
for coluna in variaveis_saneamento:
    saneamento_curitiba[coluna] = (
        saneamento_curitiba[coluna]
        .astype(str)
        .str.strip()
        .replace(["X", "", "nan"], pd.NA)
        .str.replace(",", ".", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )

In [77]:
saneamento_por_bairro = (
    saneamento_curitiba
    .groupby(["Cod_bairro", "Nome_do_bairro"], as_index=False)
    [variaveis_saneamento]
    .sum()
)

In [78]:
saneamento_por_bairro["sem_rede_geral_agua"] = (
    saneamento_por_bairro["V013"] + 
    saneamento_por_bairro["V014"] + 
    saneamento_por_bairro["V015"]
)

In [79]:
saneamento_por_bairro["sem_banheiro_sanitario"] = (
    saneamento_por_bairro["V023"]
)

In [80]:
saneamento_por_bairro["esgotamento_precario"] = (
    saneamento_por_bairro["V019"] +
    saneamento_por_bairro["V020"] +
    saneamento_por_bairro["V021"] +
    saneamento_por_bairro["V022"] +
    saneamento_por_bairro["V023"]
)

In [81]:
saneamento_por_bairro["lixo_destino_inadequado"] = (
    saneamento_por_bairro["V038"] +
    saneamento_por_bairro["V039"] +
    saneamento_por_bairro["V040"] +
    saneamento_por_bairro["V041"] +
    saneamento_por_bairro["V042"]
)

In [82]:
saneamento_por_bairro["pct_sem_rede_geral_agua"] = (
    saneamento_por_bairro["sem_rede_geral_agua"] /
    saneamento_por_bairro["V002"]
) * 100

In [83]:
saneamento_por_bairro["pct_sem_banheiro_sanitario"] = (
    saneamento_por_bairro["sem_banheiro_sanitario"] /
    saneamento_por_bairro["V002"]
) * 100

In [84]:
saneamento_por_bairro["pct_esgotamento_precario"] = (
    saneamento_por_bairro["esgotamento_precario"] /
    saneamento_por_bairro["V002"]
) * 100

In [85]:
saneamento_por_bairro["pct_lixo_destino_inadequado"] = (
    saneamento_por_bairro["lixo_destino_inadequado"] /
    saneamento_por_bairro["V002"]
) * 100

In [86]:
saneamento_bairros_final = saneamento_por_bairro[
    [
        "Cod_bairro",
        "Nome_do_bairro",
        "V002",
        "sem_rede_geral_agua",
        "pct_sem_rede_geral_agua",
        "sem_banheiro_sanitario",
        "pct_sem_banheiro_sanitario",
        "esgotamento_precario",
        "pct_esgotamento_precario",
        "lixo_destino_inadequado",
        "pct_lixo_destino_inadequado"
    ]
].rename(columns={
    "V002": "domicilios_particulares_permanentes"
})

In [87]:
saneamento_bairros_final.head()

,Cod_bairro,Nome_do_bairro,domicilios_particulares_permanentes,sem_rede_geral_agua,pct_sem_rede_geral_agua,sem_banheiro_sanitario,pct_sem_banheiro_sanitario,esgotamento_precario,pct_esgotamento_precario,lixo_destino_inadequado,pct_lixo_destino_inadequado
0,4106902001,Centro,17751,71.0,0.399977,0.0,0.000000,4.0,0.022534,0.0,0.000000
1,4106902002,São Francisco,2423,1.0,0.041271,5.0,0.206356,5.0,0.206356,0.0,0.000000
2,4106902003,Centro Cívico,2103,185.0,8.796957,0.0,0.000000,3.0,0.142653,0.0,0.000000
3,4106902004,Alto da Glória,2347,11.0,0.468683,0.0,0.000000,0.0,0.000000,1.0,0.042608
4,4106902005,Alto da Rua XV,3423,4.0,0.116857,0.0,0.000000,2.0,0.058428,0.0,0.000000


## Unificando os dados

Utilizaremos a mesma função `limpar_texto` utilizada para limpar os campos de texto da base de dados da SIGESGUARDA em `src/sigesguarda/02_limpeza_dos_dados.ipynb` objetivando padronizar os nomes dos bairros, uma vez que esses nomes serão usados como chave para a unificação dos dados do censo com os dados da sigesguarda.

Em seguida, utilizaremos as colunas `Cod_bairro` e `Nome_do_bairro` como chaves para unir os seguintes dataframes:

- `populacao_por_bairro`;
- `renda_bairros_final`;
- `alfabetizacao_bairros_final`;
- `saneamento_bairros_final`.

O resultado será salvo na pasta `data/ibge2010/cleaned`.

In [88]:
def limpar_texto(valor):
    if pd.isna(valor):
        return pd.NA
    
    valor = str(valor).strip().lower()
    
    valor = unicodedata.normalize("NFKD", valor)
    valor = valor.encode("ascii", "ignore").decode("utf-8")
    
    valor = re.sub(r"[^a-z0-9\s]", " ", valor)
    valor = re.sub(r"\s+", " ", valor).strip()
    
    return valor if valor else pd.NA

In [89]:
pop = populacao_por_bairro[
    ["Cod_bairro", "Nome_do_bairro", "populacao_2010"]
].copy()

renda = renda_bairros_final[
    [
        "Cod_bairro",
        "Nome_do_bairro",
        "pessoas_10_anos_ou_mais",
        "pct_sem_rendimento",
        "pct_rendimento_ate_1_sm",
        "pct_rendimento_ate_2_sm",
        "pct_rendimento_acima_5_sm",
        "resp_domicilios_particulares",
        "rendimento_medio_responsavel",
        "rendimento_medio_responsavel_sm"
    ]
].copy()

alf = alfabetizacao_bairros_final[
    [
        "Cod_bairro",
        "Nome_do_bairro",
        "pop_10mais",
        "alfabetizados_10mais",
        "pct_alfabetizacao_10mais",
        "pct_analfabetismo_10mais",
        "pop_15mais",
        "alfabetizados_15mais",
        "pct_alfabetizacao_15mais",
        "pct_analfabetismo_15mais",
    ]
].copy()

san = saneamento_bairros_final[
    [
        "Cod_bairro",
        "Nome_do_bairro",
        "domicilios_particulares_permanentes",
        "sem_rede_geral_agua",
        "pct_sem_rede_geral_agua",
        "sem_banheiro_sanitario",
        "pct_sem_banheiro_sanitario",
        "esgotamento_precario",
        "pct_esgotamento_precario",
        "lixo_destino_inadequado",
        "pct_lixo_destino_inadequado"
    ]
].copy()


In [90]:
base_bairros_2010 = (
    pop 
    .merge(
        renda,
        on=["Cod_bairro", "Nome_do_bairro"],
        how="outer",
        validate="1:1",
    )
    .merge(
        alf,
        on=["Cod_bairro", "Nome_do_bairro"],
        how="outer",
        validate="1:1",
    )
    .merge(
        san,
        on=["Cod_bairro", "Nome_do_bairro"],
        how="outer",
        validate="1:1",
    )
)

In [91]:
base_bairros_2010["Nome_do_bairro"] = base_bairros_2010["Nome_do_bairro"].apply(limpar_texto)

base_bairros_2010 = (
    base_bairros_2010
    .drop(columns=["Cod_bairro"])
    .rename(columns={"Nome_do_bairro": "ATENDIMENTO_BAIRRO_NOME"})
)

base_bairros_2010 = base_bairros_2010.rename(
    columns={
        coluna: f"{coluna}_2010"
        for coluna in base_bairros_2010.columns
        if coluna != "ATENDIMENTO_BAIRRO_NOME" and not
coluna.endswith("_2010")
    }
)

base_bairros_2010 = (
    base_bairros_2010
    .sort_values("ATENDIMENTO_BAIRRO_NOME")
    .reset_index(drop=True)
)

In [92]:
base_bairros_2010

,ATENDIMENTO_BAIRRO_NOME,populacao_2010,pessoas_10_anos_ou_mais_2010,pct_sem_rendimento_2010,pct_rendimento_ate_1_sm_2010,pct_rendimento_ate_2_sm_2010,pct_rendimento_acima_5_sm_2010,resp_domicilios_particulares_2010,rendimento_medio_responsavel_2010,rendimento_medio_responsavel_sm_2010,...,pct_analfabetismo_15mais_2010,domicilios_particulares_permanentes_2010,sem_rede_geral_agua_2010,pct_sem_rede_geral_agua_2010,sem_banheiro_sanitario_2010,pct_sem_banheiro_sanitario_2010,esgotamento_precario_2010,pct_esgotamento_precario_2010,lixo_destino_inadequado_2010,pct_lixo_destino_inadequado_2010
0,abranches,13151,11401.0,30.988510,13.147969,38.636962,10.174546,4067.0,2066.702583,4.052358,...,2.204097,4064,79.0,1.943898,3.0,0.073819,1016.0,25.000000,7.0,0.172244
1,agua verde,51290,47282.0,25.472696,4.445666,15.754410,36.214627,19677.0,5437.484683,10.661735,...,0.338244,19670,93.0,0.472801,0.0,0.000000,74.0,0.376207,3.0,0.015252
2,ahu,11479,10456.0,27.926549,4.982785,15.885616,35.520275,4214.0,5550.206716,10.882758,...,0.304599,4213,83.0,1.970093,0.0,0.000000,15.0,0.356041,0.0,0.000000
3,alto boqueirao,53563,46150.0,31.304442,13.087757,41.003250,5.261105,16871.0,1512.313990,2.965322,...,2.178289,16856,38.0,0.225439,8.0,0.047461,570.0,3.381585,15.0,0.088989
4,alto da gloria,5520,5163.0,25.527794,3.137711,12.880108,39.996126,2348.0,5016.747222,9.836759,...,0.343088,2347,11.0,0.468683,0.0,0.000000,0.0,0.000000,1.0,0.042608
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,uberaba,71917,61665.0,32.133301,13.805238,38.710776,9.614854,22228.0,1987.468306,3.896997,...,2.463933,22216,282.0,1.269355,22.0,0.099028,1198.0,5.392510,9.0,0.040511
71,umbara,18721,15721.0,34.482539,13.141658,42.573628,3.606641,5526.0,1314.170287,2.576804,...,3.363100,5522,225.0,4.074611,4.0,0.072438,913.0,16.533865,36.0,0.651938
72,vila izabel,11596,10558.0,27.240008,4.849403,17.058155,31.833681,4510.0,4180.128324,8.196330,...,0.398804,4503,7.0,0.155452,0.0,0.000000,19.0,0.421941,1.0,0.022207
73,vista alegre,11100,9903.0,29.758659,9.694032,28.294456,21.347067,3585.0,3740.403398,7.334124,...,1.086957,3576,31.0,0.866890,1.0,0.027964,178.0,4.977629,0.0,0.000000


In [93]:
base_bairros_2010["ATENDIMENTO_BAIRRO_NOME"].unique()

<StringArray>
[                    'abranches',                    'agua verde',
                           'ahu',                'alto boqueirao',
                'alto da gloria',                'alto da rua xv',
                         'atuba',                       'augusta',
                     'bacacheri',                   'bairro alto',
                   'barreirinha',                         'batel',
                    'bigorrilho',                     'boa vista',
                    'bom retiro',                     'boqueirao',
                  'botiatuvinha',                        'cabral',
                     'cachoeira',                        'cajuru',
           'campina do siqueira',                'campo comprido',
              'campo de santana',               'capao da imbuia',
                    'capao raso',                    'cascatinha',
                       'caximba',                        'centro',
                 'centro civico', 'cidade indust

In [94]:
ajustes_ibge2010 = {
    "botiatuvinha": "butiatuvinha",
    "cidade industrial de curitiba": "cidade industrial",
    "alto da rua xv": "alto da xv",
    "campo de santana": "campo do santana"
}

In [95]:
base_bairros_2010["ATENDIMENTO_BAIRRO_NOME"] = (
    base_bairros_2010["ATENDIMENTO_BAIRRO_NOME"]
    .apply(limpar_texto)
    .replace(MAPA_BAIRRO)
    .replace(ajustes_ibge2010)
)

In [96]:
base_filtrada = base_bairros_2010[base_bairros_2010['ATENDIMENTO_BAIRRO_NOME'] == 'cidade industrial']

In [97]:
base_filtrada

,ATENDIMENTO_BAIRRO_NOME,populacao_2010,pessoas_10_anos_ou_mais_2010,pct_sem_rendimento_2010,pct_rendimento_ate_1_sm_2010,pct_rendimento_ate_2_sm_2010,pct_rendimento_acima_5_sm_2010,resp_domicilios_particulares_2010,rendimento_medio_responsavel_2010,rendimento_medio_responsavel_sm_2010,...,pct_analfabetismo_15mais_2010,domicilios_particulares_permanentes_2010,sem_rede_geral_agua_2010,pct_sem_rede_geral_agua_2010,sem_banheiro_sanitario_2010,pct_sem_banheiro_sanitario_2010,esgotamento_precario_2010,pct_esgotamento_precario_2010,lixo_destino_inadequado_2010,pct_lixo_destino_inadequado_2010
29,cidade industrial,171930,147394.0,31.466681,14.002605,45.06832,3.5802,53010.0,1322.486476,2.593111,...,3.268855,52977,102.0,0.192536,44.0,0.083055,1295.0,2.444457,28.0,0.052853


In [98]:
output_path = data_dir / "cleaned/base_bairros_2010.csv"

In [99]:
base_bairros_2010.to_csv(output_path, index=False, encoding="latin1")